## Load StackOverflow Dataset

In [17]:
import pandas as pd

dfQuestions = pd.read_csv('Questions.csv',
                            encoding = "ISO-8859-1",
                            usecols = ['Id','Score','Title'])
#answers table
dfAnswers = pd.read_csv('Answers.csv',
                            encoding = "ISO-8859-1",
                            usecols = ['ParentId','Score','Body'],#parent id links to the questions table
                            )

dfQuestions = dfQuestions[dfQuestions['Score'] > 0]
dfAnswers = dfAnswers[dfAnswers['Score'] > 0]\
    .sort_values('Score',ascending=False)\
    .drop_duplicates(subset=['ParentId'])

qaDf = dfQuestions.merge(dfAnswers,left_on = 'Id', right_on = 'ParentId')\
    .rename(columns={'Title':'Question','Body':'Answer'})[['Question','Answer','Score_x']]

# stackQaData = []
# for index, row in qaDf.iterrows():
#     stackQaData.append(f"Question:\n{row['Question']}\n\nAnswer:\n{row['Answer']}")
    
qaDf['text'] = 'Question:\n' + qaDf['Question'] + '\n\nAnswer:\n' + qaDf['Answer']

In [18]:
qaDf.head()

,Question,Answer,Score_x,text
0,How can I find the full path to a font from it...,<p>Unfortunately the only API that isn't depre...,21,Question:\nHow can I find the full path to a f...
1,Get a preview JPEG of a PDF on Windows?,<p>ImageMagick delegates the PDF->bitmap conve...,27,Question:\nGet a preview JPEG of a PDF on Wind...
2,Continuous Integration System for a Python Cod...,<p>One possibility is Hudson. It's written in...,40,Question:\nContinuous Integration System for a...
3,cx_Oracle: How do I iterate over a result set?,<p>The canonical way is to use the built-in cu...,25,Question:\ncx_Oracle: How do I iterate over a ...
4,Using 'in' to match an attribute of Python obj...,<p>Using a list comprehension would build a te...,28,Question:\nUsing 'in' to match an attribute of...


### Preprocess Stack Overflow data

In [19]:
from bs4 import BeautifulSoup

def preprocessText(text):
    # replace <code>sample code</code> with ``` tags
    text = text.replace("<code>", "```").replace("</code>", "```")
    soup = BeautifulSoup(text, 'html.parser')
    text = soup.get_text()   
    del soup
    return text

In [20]:
# raw data
raw_text = qaDf['text'][0]
print("Before Preprocessing")
print(raw_text)

print()
print("After Preprocessing")
print(preprocessText(raw_text))

Before Preprocessing
Question:
How can I find the full path to a font from its display name on a Mac?

Answer:
<p>Unfortunately the only API that isn't deprecated is located in the ApplicationServices framework, which doesn't have a bridge support file, and thus isn't available in the bridge. If you're wanting to use ctypes, you can use ATSFontGetFileReference after looking up the ATSFontRef.</p>

<p>Cocoa doesn't have any native support, at least as of 10.5, for getting the location of a font.</p>

After Preprocessing
Question:
How can I find the full path to a font from its display name on a Mac?

Answer:
Unfortunately the only API that isn't deprecated is located in the ApplicationServices framework, which doesn't have a bridge support file, and thus isn't available in the bridge. If you're wanting to use ctypes, you can use ATSFontGetFileReference after looking up the ATSFontRef.
Cocoa doesn't have any native support, at least as of 10.5, for getting the location of a font.


In [21]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

In [22]:
import os
num_cores = os.cpu_count()
NUM_WORKERS = num_cores//2 + 1

In [23]:
from tqdm.auto import tqdm
from joblib import Parallel, delayed
import warnings

warnings.simplefilter("ignore")

# Parallelize the preprocessing function
qaDf['text'] = Parallel(n_jobs=NUM_WORKERS)(
    delayed(preprocessText)(text) for text in qaDf['text'])

In [24]:
qaDf['text']

0         Question:\nHow can I find the full path to a f...
1         Question:\nGet a preview JPEG of a PDF on Wind...
2         Question:\nContinuous Integration System for a...
3         Question:\ncx_Oracle: How do I iterate over a ...
4         Question:\nUsing 'in' to match an attribute of...
                                ...                        
262799    Question:\nDifferent result for String and Int...
262800    Question:\ncan't assign to function call Error...
262801    Question:\nerror handling with BeautifulSoup w...
262802    Question:\nfinding cubed root using delta and ...
262803    Question:\nHow to execute multiline python cod...
Name: text, Length: 262804, dtype: str

In [ ]:
fin_list = qaDf['text'].to_list()
fin_list[0:5]

["Question:\nHow can I find the full path to a font from its display name on a Mac?\n\nAnswer:\nUnfortunately the only API that isn't deprecated is located in the ApplicationServices framework, which doesn't have a bridge support file, and thus isn't available in the bridge. If you're wanting to use ctypes, you can use ATSFontGetFileReference after looking up the ATSFontRef.\nCocoa doesn't have any native support, at least as of 10.5, for getting the location of a font.",
 "Question:\nGet a preview JPEG of a PDF on Windows?\n\nAnswer:\nImageMagick delegates the PDF->bitmap conversion to GhostScript anyway, so here's a command you can use (it's based on the actual command listed by the ```ps:alpha``` delegate in ImageMagick, just adjusted to use JPEG as output):\n```gs -q -dQUIET -dPARANOIDSAFER -dBATCH -dNOPAUSE -dNOPROMPT \\\n-dMaxBitmap=500000000 -dLastPage=1 -dAlignToPixels=0 -dGridFitTT=0 \\\n-sDEVICE=jpeg -dTextAlphaBits=4 -dGraphicsAlphaBits=4 -r72x72 \\\n-sOutputFile=$OUTPUT -f$

In [27]:
len(fin_list)

262804

In [26]:
import json
with open("question_answer.json", "w") as file:
    json.dump(fin_list, file)